In [2]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

import time
import json
import ctypes
import struct
import blosc2
import numpy as np
import numpy as np
from pathlib import Path
import sys

In [3]:
sys.path.append('/vast/projects/exasky/pascal/HACC/venv_seerx/lib/python3.13/site-packages')

In [4]:
sys.path.append('/vast/projects/exasky/pascal/HACC/SZ3/tools/pysz')
from pysz import SZ

In [5]:
import pyvista as pv

ModuleNotFoundError: No module named 'pyvista'

In [6]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [7]:
def list_keys(db, simid):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            x.append(k)

    return x

In [8]:
def list_fields(db, simid):
    
    all_keys = list_all_keys(db)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-2]
                x.append(name)
        
    return list(set(x))

In [9]:
def list_attributes(db, simid):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        if parts[0] == '_' + str(simid):
            if len(parts) == 5:
                name = parts[len(parts)-1]
                x.append(name)

    return list(set(x))

In [10]:
def put_value(db, key, value):
    ''' Put data into the server for that key '''
    db.put(key=key, value=value)  # get the data


In [11]:
def get_raw_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)
    #print(l)
    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    #print(type(out_val))
    return out_val

In [12]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [13]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [14]:
def get_decompDataBLOSC(db, key, num_elems):
        x = []
        
        val = get_data(db, key)
        a_bytesobj2 = blosc2.decompress(val)
        
        bf = str(num_elems) + 'f'
        x = struct.unpack(bf, a_bytesobj2)

        return x

In [15]:
def get_decompDataSZ3(db, key, num_elems):
        data = get_data(db, key)
        np_array = np.frombuffer(data, dtype=np.uint8)
        
        lib_extention = {
            "darwin": "libSZ3c.dylib",
            "windows": "SZ3c.dll",
        }.get(sys.platform, "libSZ3c.so")

        sz = SZ("/projects/insituperf/SZ3/install/lib64/{}".format(lib_extention))
        
        data_dec = sz.decompress(np_array, (num_elems,1,1), np.float32)


        return data_dec

In [16]:
def isTsReady(db, ts, simid):
    key = '_' + simid + '/' + ts +'/status'
    return get_value(db, key)

In [17]:
def getNumRanks(db, simid):
    key = '_' + simid + '/num_ranks'
    return get_value(db, key)

In [18]:
f = open('/vast/projects/exasky/pascal/HACC/Seer/simple_app/mochi-yokan-config.json')
#f = open('/vast/projects/exasky/pascal/HACC/trunk/mochi-yokan-config.json')

In [19]:
json_data = json.load(f)
json_data

{'sim-id': '8',
 'benchmark': {'num-ts': 50, 'num-elements': 200000},
 'libraries': ['/vast/home/pascalgrosset/software/spack/opt/spack/linux-broadwell/mochi-yokan-0.8.1-njadzndq6fkjnrva3utxazt4psj6pc2v/lib/libyokan-bedrock-module.so'],
 'providers': [{'name': 'yokan_provider',
   'provider_id': 124,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map', 'config': {}}}}],
 'data': [{'name': 'pressure_3', 'compressor': 'BLOSC'},
  {'name': 'temperature_3', 'compressor': 'SZ3', 'mode': 'psnr', 'value': 50},
  {'name': 'energy_3', 'compressor': 'SZ3', 'mode': 'psnr', 'value': 75}],
 'databases': [{'address': '192.168.81.74:37279',
   'protocol': 'ofi+tcp',
   'provider_id': 124}]}

In [20]:
server_addr1 = "ofi+tcp://192.168.81.74:37279"
provider_id = 124
protocol = 'ofi+tcp'

In [21]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [ ]:
server_addr2 = "ofi+tcp://192.168.81.78:45745"
provider_id = 124
protocol = 'ofi+tcp'

In [ ]:
engine2 = Engine(protocol)
mid2 = engine2.get_internal_mid()
addr2 = engine2.lookup(server_addr2)
hg_addr2 = addr2.get_internal_hg_addr()
provider2 = Provider(mid=mid2, provider_id=provider_id, config='{"database":{"type":"map"}}')
client2 = Client(mid=mid2)
db2 = client2.make_database_handle(address=hg_addr2, provider_id=provider_id)

In [22]:
dbs = []

In [23]:
dbs.append(db1)


In [ ]:
dbs.append(db2)

In [ ]:
arr = np.random.rand(10)
print(arr)
print(type(arr[0]))

a_bytes = arr.tobytes()
print(type(a_bytes))

aa_bytes = bytearray(arr)
print(type(aa_bytes))


print(len(aa_bytes))
put_value(db1, "1", aa_bytes)

y = np.frombuffer(aa_bytes, dtype=np.float64)
print(y)


# print(len(a_bytes))
# put_value(db1, "1", a_bytes)

# y = np.frombuffer(a_bytes, dtype=np.float64)
# print(y)

In [ ]:
x = get_raw_value(db1, "1")
y = np.frombuffer(x, dtype=np.float64)
y

In [ ]:
print("num_elements, size, time")
for i in range(50):
    num_elements = 2000000*i
    arr = np.random.rand(num_elements)
    aa_bytes = bytearray(arr)
    print(num_elements, len(aa_bytes))

In [ ]:
from sys import getsizeof

print("num_elements, size, time")
for i in range(50):
    num_elements = 2000000*i
    arr = np.random.rand(num_elements)
    aa_bytes = bytearray(arr)

    start = time.time()
    put_value(db1, str(i), aa_bytes)
    end = time.time()
    #print("time" , end - start, "num_elements:", num_elements, "size:", (getsizeof(arr)))
    print(num_elements, ",", len(aa_bytes), ",", (end - start))



In [ ]:
#get_value(dbs[0],"1")

In [ ]:
print("# , time")
for i in range(50):
    start = time.time()
    x = get_raw_value(db1, str(i))
    end = time.time()
    print(i, ",", (end - start))

In [ ]:
ts = '0'

In [24]:
simid = '8'

In [ ]:
keys = list_all_keys(dbs[0])
keys

In [ ]:
_<sim_id>/<ts>/<rank>/<item>/<properties>

In [25]:
list_keys(dbs[0],simid)

['_8/0/0/energy_3/compressed_size',
 '_8/0/0/energy_3/dbIndex',
 '_8/0/0/energy_3/num_elems',
 '_8/0/0/energy_3/type',
 '_8/0/0/energy_3/value',
 '_8/0/0/pressure_3/compressed_size',
 '_8/0/0/pressure_3/dbIndex',
 '_8/0/0/pressure_3/num_elems',
 '_8/0/0/pressure_3/type',
 '_8/0/0/pressure_3/value',
 '_8/0/0/status',
 '_8/0/0/temperature_3/compressed_size',
 '_8/0/0/temperature_3/dbIndex',
 '_8/0/0/temperature_3/num_elems',
 '_8/0/0/temperature_3/type',
 '_8/0/0/temperature_3/value',
 '_8/0/simDone',
 '_8/1/0/energy_3/compressed_size',
 '_8/1/0/energy_3/dbIndex',
 '_8/1/0/energy_3/num_elems',
 '_8/1/0/energy_3/type',
 '_8/1/0/energy_3/value',
 '_8/1/0/pressure_3/compressed_size',
 '_8/1/0/pressure_3/dbIndex',
 '_8/1/0/pressure_3/num_elems',
 '_8/1/0/pressure_3/type',
 '_8/1/0/pressure_3/value',
 '_8/1/0/status',
 '_8/1/0/temperature_3/compressed_size',
 '_8/1/0/temperature_3/dbIndex',
 '_8/1/0/temperature_3/num_elems',
 '_8/1/0/temperature_3/type',
 '_8/1/0/temperature_3/value',
 '_8/10

In [26]:
fields = list_fields(dbs[0], simid)
fields

['temperature_3', 'pressure_3', 'energy_3']

In [27]:
attributes = list_attributes(dbs[0], simid)
attributes

['compressed_size', 'num_elems', 'type', 'value', 'dbIndex']

In [28]:
isTsReady(dbs[0], '1', simid)

Exception: Key not found

In [31]:
get_value(dbs[0], f"_{simid}/0/0/status")

'done'

In [ ]:
getNumRanks(dbs[0], simid)

In [ ]:
get_value(dbs[0], "_07720/0/1/pressure_3/dbIndex")

In [ ]:
get_value(dbs[0], "_07720/1/1/pressure_3/dbIndex")

In [37]:
nE = get_value(dbs[0],f"_{simid}/0/0/pressure_3/num_elems")
nE

'200000'

In [33]:
get_value(dbs[0],f"_{simid}/0/0/temperature_3/value")

UnicodeDecodeError: 'ascii' codec can't decode byte 0xf3 in position 1: ordinal not in range(128)

In [38]:
get_decompDataBLOSC(dbs[0],f"_{simid}/0/0/pressure_3/value", 200000)

(2.130385637283325,
 0.9807865023612976,
 4.6147356033325195,
 0.4363574981689453,
 0.501425564289093,
 0.7591336369514465,
 0.7105255722999573,
 1.0393307209014893,
 0.8862601518630981,
 0.23329459130764008,
 0.06711181998252869,
 0.1706485152244568,
 0.3907751441001892,
 1.1928303241729736,
 4.576828956604004,
 0.6112799048423767,
 2.0695080757141113,
 1.216036319732666,
 0.5073968768119812,
 0.3792232871055603,
 0.6841608881950378,
 0.44892361760139465,
 0.8038247227668762,
 0.8788914680480957,
 0.07347866147756577,
 6.112256050109863,
 0.28981611132621765,
 2.5515925884246826,
 3.2048728466033936,
 7.2540717124938965,
 0.2454889565706253,
 1.0602740049362183,
 0.4940447211265564,
 0.49353188276290894,
 0.7675859332084656,
 13.533650398254395,
 0.46965134143829346,
 1.2910836935043335,
 0.3849427103996277,
 1.8076013326644897,
 4.144388198852539,
 0.5000759959220886,
 2.5101661682128906,
 0.25629645586013794,
 0.5102308988571167,
 1.624877691268921,
 0.7654852271080017,
 0.475729733

In [ ]:
pressure_3 = get_decompDataSZ3(dbs[0], f"_{simid}/0/0/pressure_3/value", 200000)
pressure_3

In [ ]:
pressure_3 = get_decompDataBLOSC(dbs[1], "_07720/1/1/pressure_3/value", 100)
pressure_3

In [ ]:
len(com_x)

In [ ]:
get_value(db, "_56789/499/0/x/num_elems")

In [ ]:
val_x = get_decompDataSZ3(db, "_56789/499/0/x/value", 3080753)
vals_x = val_x.flatten()

In [ ]:
val_y = get_decompDataSZ3(db, "_56789/499/0/y/value", 3080753)
vals_y = val_y.flatten()

In [ ]:
val_z = get_decompDataSZ3(db, "_56789/499/0/z/value", 3080753)
vals_z = val_z.flatten()

In [ ]:
xxx= np.stack([vals_x,vals_y,vals_z], axis=1)

In [ ]:
import pyvista as pv
from pyvista import examples
from pyvista.trame.jupyter import elegantly_launch, launch_server

pv.global_theme.trame.server_proxy_enabled = True
pv.global_theme.trame.server_proxy_prefix = 'darwin-fe.lanl.gov:8879' # white screen

points = xxx
point_cloud = pv.PolyData(points)
point_cloud.plot(jupyter_backend='trame', eye_dome_lighting=True)
point_cloud.point_size = 0.01
point_cloud.opacity = 0.1

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(point_cloud,opacity=0.25,point_size=1.0,color='#0000ff')
plotter.camera.zoom(4.0)
plotter.show(jupyter_backend='static') #works screen

In [ ]:
np.savetxt("/projects/insituperf/seer_o/3d_array.csv", xxx, delimiter=",")

In [ ]:
np.save('/projects/insituperf/seer_o/my_array.npy', xxx)

In [ ]:
import pyvista as pv

In [ ]:
%pip list

In [ ]:
!{sys.executable} -m pip install 'pyvista[jupyter]>=0.38.1'